In [6]:
import numpy as np
import pandas as pd
import sklearn
import sentence_transformers
import chromadb

np.set_printoptions(precision=3, suppress=True)
print("numpy", np.__version__)
print("sentence_transformers", sentence_transformers.__version__)
print("chromadb", chromadb.__version__)


numpy 2.4.6
sentence_transformers 5.5.1
chromadb 1.1.1


# Vectors
- list of nums
---
# Embedding models
- turn text into a vector and texts with similar meaning end up with vectors that point in similar directions
- measurements(distance) are required in order to compare them
---
# Metrics
**Measure** | **What it captures** | **Range** | **Higher val implies**
|---|---|---|---|
| **Cosine Similarity** | angle(direction) only | -1,1 | more similar | 
| **Dot Product** | angle(direction) + length | unbound | more similar | 
| **Euclidean (L2) Distance** | straight-line gap | 0 to $\infty$ | less similar | 

- cosine is the usual choice for text because the *length* of an embedding usually carries no meaning, only direction does
- formula: $\cos(a, b) = \frac{a \cdot b}{\|a\| \times \|b\|}$


In [7]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 4.0, 6.0]) # same direction as a, just 2x longer than a
c = np.array([-3.0, 0.5, 1.0]) # points in a different direction

print("dot(a, b): ", np.dot(a, b))
print("norm(a)  : ", np.linalg.norm(a)) # length = sqrt(sum of squares)
print("norm(b)  : ", np.linalg.norm(b))

dot(a, b):  28.0
norm(a)  :  3.7416573867739413
norm(b)  :  7.483314773547883


In [8]:
def cosine_sim(x, y):
    """cosine similarity between 2 1-D  vectors"""
    return np.dot(x,y) / (np.linalg.norm(x) * np.linalg.norm(y))

def l2_dist(x,  y):
    """euclidean distance between 2 1-D vectors"""
    return np.linalg.norm(x - y)

print("cosine(a, b): ", cosine_sim(a, b))
print("cosine(a, c): ", cosine_sim(a, c))
print("l2(a, b): ", l2_dist(a, b))
print("l2(a, c): ", l2_dist(a, c))


cosine(a, b):  1.0
cosine(a, c):  0.08347838711296822
l2(a, b):  3.7416573867739413
l2(a, c):  4.716990566028302


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

print("manual:      ", cosine_sim(a, c))
print("sklearn:     ", cosine_similarity(a.reshape(1, -1), c.reshape(1, -1))[0, 0])

manual:       0.08347838711296822
sklearn:      0.08347838711296829


### Trick: 
- normalize once, then dot product == cosine
- if every vector has len 1, the denominator in the formula becomes 1 thus the cosine similarity is just the dot product
- many systems normalize the embeddings up front:
    - dot products are fast
    - cosine is free once the denominator is 1

In [11]:
def normalize(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

a_n, c_n = normalize(a), normalize(c)
print("norm after normalizing:", np.linalg.norm(a_n))
print("dot of unit vectors   :", np.dot(a_n, c_n))
print("cosine of originals   :", cosine_sim(a, c))

norm after normalizing: 1.0
dot of unit vectors   : 0.08347838711296829
cosine of originals   : 0.08347838711296822


In [12]:
# similarity of many vectors at once (what a vector DB does at scale)
rng = np.random.default_rng(42)
query = rng.normal(size=8)
docs = rng.normal(size=(5, 8))          # 5 "documents", 8 dimensions each

scores = normalize(docs) @ normalize(query)     # one matrix multiply
print("scores     :", scores)
print("best match :", scores.argmax())
print("ranking    :", np.argsort(-scores))      # indices, best first

scores     : [ 0.169  0.564  0.002  0.535 -0.255]
best match : 1
ranking    : [1 3 0 2 4]


## Embeddings

### Concept
An **embedding model** is a neural network trained so that similar meanings produce nearby vectors. We use `all-MiniLM-L6-v2`: small, fast, free, runs on a CPU, outputs **384 dimensions**.

Rule you must never break: **embed your documents and your queries with the same model.** Vectors from different models live in different spaces and cannot be compared.

In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("dimensions: ", model.get_embedding_dimension())
print("max tokens: ", model.max_seq_length)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\premt\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\premt\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

dimensions:  384
max tokens:  256


C:\Users\premt\AppData\Local\Temp\ipykernel_21596\169615132.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("dimensions: ", model.get_sentence_embedding_dimension())


In [14]:
sentences = [
    "I am vengeance",
    "I am the night",
    "I am batman",
    "Gotham needs help, the night is louder than usual.",
    "Alfred, get me joker's latest whereabouts."
]

emb = model.encode(sentences, normalize_embeddings=True)
print("shape:", emb.shape)
print("first 8 numbers of sentence 0: ", emb[0][:8])

shape: (5, 384)
first 8 numbers of sentence 0:  [-0.072  0.034  0.039 -0.033 -0.042 -0.021  0.067 -0.004]
